In [34]:
import pandas as pd
import numpy as np


terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")

C:\Users\HP\AppData\Local\Temp\ipykernel_5848\551476585.py:5: DtypeWarning: Columns (4,6,31,33,61,62,63,76,79,90,92,94,96,114,115,121) have mixed types. Specify dtype option on import or set low_memory=False.
  terrorism = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\Global Terrorism Dataset\globalterrorismdb_0718dist.csv", encoding="Windows-1252")


In [35]:
replace_dict = {
    "unknown":pd.NA,
    "maoists":"communist party of india - maoist (cpi-maoist)",
    "palestinian extremists":"palestine liberation organization (plo)",
    "palestinians": "palestine liberation organization (plo)", 
    "muslim militants":"muslim extremists",
    "muslim separatists":"muslim extremists",
    "muslim fundamentalists":"muslim extremists",
    "muslim rebels":"muslim extremists",
    "muslims":"muslim extremists",
    "islamic state of iraq (isi)":"islamic state of iraq and the levant (isil)" 
}

In [36]:
#Run this cell only once. The conversion of unknown values to NaN leads to them being dropped on a second run.

terrorism["gname"] =(
    terrorism["gname"]
    .str.strip()
    .str.lower()
    .replace(to_replace=replace_dict)
) 


ranks = (
    terrorism
    .groupby("gname")["eventid"]
    .count()
    .sort_values(ascending=False)
)

ranks = ranks[ranks>30]
terrorism = terrorism[terrorism["gname"].isin(ranks.index)]

with open(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\ranks.txt", "w") as file_rank:
    file_rank.write(ranks.to_string())



In [37]:
default_col = [
    "target",
    "ransom",
    "targ",
    "corp",
    "natlty",
    "guncertain",
    "claimmode",
    "weap",
    "ishostkid",
    "scite",
    "hostkidoutcome",
    "prop",
    "nperp"
]
additional_columns = [
    "attacktype2_txt",
    "attacktype2",
    "attacktype3",
    "attacktype3_txt",
    "gsubname2",
    "gsubname3",
    "gname2",
    "gname3",
    "dbsource",
    "addnotes",
    "nkillus",
    "nwoundus",
    "nkillter",
    "nwoundte",
    "location" 
]

In [38]:

collist = [col for col in terrorism.columns if any(w in col.lower() for w in default_col)] + additional_columns
    

terrorism =terrorism.drop(columns=collist) 

terrorism = terrorism.drop(columns= [col for col in terrorism.columns if (terrorism[col].isna().mean())>0.85])
terrorism.columns

Index(['eventid', 'iyear', 'imonth', 'iday', 'extended', 'country',
       'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude',
       'longitude', 'specificity', 'vicinity', 'summary', 'crit1', 'crit2',
       'crit3', 'doubtterr', 'alternative', 'alternative_txt', 'multiple',
       'success', 'suicide', 'attacktype1', 'attacktype1_txt', 'gname',
       'motive', 'individual', 'claimed', 'nkill', 'nwound', 'INT_LOG',
       'INT_IDEO', 'INT_MISC', 'INT_ANY', 'related'],
      dtype='object')

In [39]:
terrorism.isna().sum()

eventid                0
iyear                  0
imonth                 0
iday                   0
extended               0
country                0
country_txt            0
region                 0
region_txt             0
provstate            116
city                  93
latitude            2849
longitude           2849
specificity            3
vicinity               0
summary            36962
crit1                  0
crit2                  0
crit3                  0
doubtterr              0
alternative        70785
alternative_txt    70785
multiple               1
success                0
suicide                0
attacktype1            0
attacktype1_txt        0
gname                  0
motive             65345
individual             0
claimed            36953
nkill               6333
nwound             10783
INT_LOG                0
INT_IDEO               0
INT_MISC               0
INT_ANY                0
related            70960
dtype: int64

In [40]:
#Date Parsing code
terrorism["date"] = terrorism.iyear.astype(str) + "/" + terrorism.imonth.astype(str) + "/" + terrorism.iday.astype(str)
terrorism["date"] = pd.to_datetime(terrorism["date"], format=f"%Y/%#m/%#d", errors="coerce")

terrorism = terrorism.dropna(subset="date")
terrorism = terrorism.drop(columns=["imonth", "iyear", "iday"])


In [41]:
cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")

cities["City"] =(
    cities["City"]
    .str.lower()
    .str.strip()
) 
cities = cities.drop_duplicates(subset=["City"])

terrorism["city"] =(
    terrorism["city"]
    .str.lower()
    .str.strip()
) 

terrorism = pd.merge(terrorism, cities, how="left", left_on="city", right_on="City")

terrorism["latitude"] = terrorism.latitude.fillna(terrorism.Latitude)
terrorism["longitude"] = terrorism.longitude.fillna(terrorism.Longitude)


C:\Users\HP\AppData\Local\Temp\ipykernel_5848\279449341.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  cities = pd.read_csv(f"Global Terrorism Dataset/worldcitiespop.csv")


In [42]:
drops = ["Country","City","AccentCity","Region","Population","Latitude","Longitude"]
terrorism = terrorism.drop(columns= drops)

In [43]:
analytics =(
    terrorism
    .groupby(["gname", "country_txt"], as_index=False)
    .filter(lambda x:x["nkill"].sum() > 20 )
    .groupby(["gname", "country_txt"], as_index=False)
    .count()
    .sort_values("nkill", ascending=False)
    .sort_values(by="country_txt", kind="stable")
    .set_index(["country_txt", "gname"])
    
)
analytics_txt = analytics[["nkill","nwound"]].to_string()
with open(r"C:\Users\HP\OneDrive\Desktop\Code\Hackclub\TerrorAnalyzer\analytics.txt", "w") as f:
    f.write(analytics_txt)
    f.close()
